# LangChain for GenAI Applications

We will use LangChain to connect prompts, models, tools, and chains. The main coding focus is **tool calling with LangChain**.

## Learning Objectives

By the end of this notebook, you should be able to:

1. Explain what LangChain is.
2. Use a LangChain chat model.
3. Define LangChain tools.
4. Let an LLM request tools.
5. Execute tool calls from Python.
6. Send tool results back to the model.
7. Use one and two tools for math operations.
8. Build a small LangChain chain.
9. Understand a generic LangChain tool-calling loop.

## What Is LangChain?

LangChain is a framework for building applications with LLMs.

It helps you organize common building blocks:

- chat models
- prompts
- tools
- chains
- agents
- retrievers
- memory
- output parsers

Simple LangChain application:

```text
User input
    |
    v
Prompt
    |
    v
Chat model
    |
    v
Output
```

Tool-using LangChain application:

```text
User question
    |
    v
Chat model with tools
    |
    +-- final answer --------------------+
    |                                    |
    +-- tool call request                |
            |                            |
            v                            |
        Python executes tool             |
            |                            |
            v                            |
        Tool result sent to model -------+
            |
            v
        Final answer
```

## Why Tools Matter

LLMs normally generate text.

Tools allow an LLM application to interact with external systems.

Examples of real tools:

- search a knowledge base
- query a database
- call an internal API
- create a ticket
- check order status
- calculate a value

In this notebook, tools are **mock tools**. They are normal Python functions. This keeps the class safe, simple, and focused.

## Setup

We will use:

- `openai`: the official OpenAI Python SDK
- `langchain`
- `langchain-openai`: LangChain's OpenAI integration
- `pydantic`: a dependency used by LangChain for schemas and validation

The API key should be available as:

```text
OPENAI_API_KEY
```

Suggested model:

```python
MODEL = "gpt-4.1-mini"
```

## Check Environment

```python
OPENAI_API_KEY
```

The next cell checks that the API key exists and prints the model name used in the notebook.

In [42]:

import os, sys, importlib, types

# ── Load environment (LiteLLM proxy) ────────────────────────────────────────
sys.path.insert(0, os.path.join(os.getcwd(), "utils"))

# Load .env for LITELLM_BASE_URL / LITELLM_API_KEY / MODEL
try:
    import dotenv
    dotenv.load_dotenv()
except ImportError:
    pass

os.environ.setdefault("LITELLM_BASE_URL", "https://genailab.tcs.in")
os.environ.setdefault("LITELLM_API_KEY",  "sk-YVGrJjZiUDuNCB67oXQBDQ")
os.environ.setdefault("MODEL",            "gpt-4.1-mini")
os.environ.setdefault("OPENAI_API_KEY",   os.environ["LITELLM_API_KEY"])

MODEL = os.environ["MODEL"]

# ── Load our fake OpenAI module ──────────────────────────────────────────────
import utils.open_ai as _oai_mod
importlib.reload(_oai_mod)
from utils.open_ai import OpenAI, ChatOpenAI

# Patch sys.modules so any import resolves to our fakes:
#   from openai import OpenAI           → our OpenAI
#   from langchain_openai import ChatOpenAI → our ChatOpenAI
_fake_openai = types.ModuleType("openai")
_fake_openai.OpenAI = OpenAI
_fake_openai.__version__ = "fake-litellm"
sys.modules["openai"] = _fake_openai

_fake_lc_openai = types.ModuleType("langchain_openai")
_fake_lc_openai.ChatOpenAI = ChatOpenAI
sys.modules["langchain_openai"] = _fake_lc_openai

# Register ChatOpenAI as a virtual Runnable so LCEL pipe (prompt | llm) works
try:
    from langchain_core.runnables import Runnable as _LCRunnable
    _LCRunnable.register(ChatOpenAI)
    from utils.open_ai import _BoundChatOpenAI
    _LCRunnable.register(_BoundChatOpenAI)
except Exception:
    pass

print("Environment is ready (routing via LiteLLM proxy).")
print("Model:", MODEL)


Environment is ready (routing via LiteLLM proxy).
Model: gpt-4.1-mini


## The Official OpenAI SDK Is Still Underneath

LangChain's OpenAI integration uses the official OpenAI Python SDK underneath. LangChain adds structure around prompts, models, tools, chains, and agents.

Direct SDK style:

```text
Your code -> OpenAI SDK -> OpenAI model
```

LangChain style:

```text
Your code -> LangChain model wrapper -> OpenAI SDK -> OpenAI model
```

In [29]:
import openai
from openai import OpenAI

print("OpenAI SDK version:", openai.__version__)

# We create a client to show that the official SDK is available.
# LangChain's ChatOpenAI integration uses this SDK internally.
sdk_client = OpenAI()
print("Official OpenAI SDK client is ready.")

OpenAI SDK version: fake-litellm
Official OpenAI SDK client is ready.


## Create a LangChain Chat Model

`ChatOpenAI` is LangChain's chat model wrapper for OpenAI models.

We set a low temperature so live demos are more predictable.

In [43]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=MODEL,
    temperature=0,
)

response = llm.invoke("In one sentence, explain LangChain to a beginner.")
print(response.content)

LangChain is a tool that helps developers build applications by connecting language models with other data sources and tools to create smarter, more interactive programs.


## LangChain Messages

LangChain represents conversations with message objects.

Common message types:

- `SystemMessage`: instructions for the model
- `HumanMessage`: user input
- `AIMessage`: model output
- `ToolMessage`: result returned by a tool

Message flow:

```text
SystemMessage -> HumanMessage -> AIMessage -> ToolMessage -> AIMessage
```

In [31]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="You are a concise GenAI instructor."),
    HumanMessage(content="What problem does LangChain solve? Answer in one short paragraph."),
]

response = llm.invoke(messages)
print(response.content)

LangChain solves the problem of building applications that effectively integrate large language models (LLMs) with external data sources, APIs, and user interactions by providing a framework to manage prompts, chains, memory, and agents, enabling developers to create more dynamic, context-aware, and functional AI-powered applications.


## LangChain Tools

In LangChain, a tool is usually a Python function with a name, description, and input schema.

The easiest way to define a tool is with the `@tool` decorator.

LangChain uses the function name, docstring, and type hints to describe the tool to the model.

In [32]:
from langchain_core.tools import tool


@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers and return the result."""
    return a + b


print("Tool name:", add_numbers.name)
print("Tool description:", add_numbers.description)
print("Tool args schema:")
print(add_numbers.args)

Tool name: add_numbers
Tool description: Add two numbers and return the result.
Tool args schema:
{'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}


## Bind a Tool to the Model

The model does not automatically know about our Python functions.

We use `bind_tools(...)` to tell the chat model what tools are available.

```text
Python tool
    |
    v
bind_tools
    |
    v
Chat model can request that tool
```

In [33]:
llm_with_add_tool = llm.bind_tools([add_numbers])

messages = [
    SystemMessage(content="You are careful with math. Use tools when calculations are needed."),
    HumanMessage(content="What is 125 + 378? Use the available tool."),
]

ai_message = llm_with_add_tool.invoke(messages)

print("Text content:", ai_message.content)
print("Tool calls:")
print(ai_message.tool_calls)

if not ai_message.tool_calls:
    raise RuntimeError(
        "Expected the model to request add_numbers, but it returned no tool calls. "
        "Re-run this cell or make the user request more explicit."
    )

Text content: 
Tool calls:
[{'name': 'add_numbers', 'args': {'a': 125, 'b': 378}, 'id': 'call_mBoNHsYH1eAXEC36MWcQPwAa'}]


## Execute a LangChain Tool Call

Important idea:

The model can request a tool, but Python executes it.

The model response contains a structured tool call:

```text
tool name + arguments + tool call id
```

Then our Python code runs the matching tool.

In [34]:
tool_call = ai_message.tool_calls[0]

print("Tool requested:", tool_call["name"])
print("Arguments:", tool_call["args"])
print("Tool call id:", tool_call["id"])

tool_result = add_numbers.invoke(tool_call["args"])
print("Tool result:", tool_result)

Tool requested: add_numbers
Arguments: {'a': 125, 'b': 378}
Tool call id: call_mBoNHsYH1eAXEC36MWcQPwAa
Tool result: 503.0


## Send Tool Results Back with `ToolMessage`

After executing the tool, we send the result back to the model using `ToolMessage`.

The `tool_call_id` connects the result to the original tool request.

In [35]:
from langchain_core.messages import ToolMessage

messages_with_tool_result = messages + [
    ai_message,
    ToolMessage(
        content=str(tool_result),
        tool_call_id=tool_call["id"],
    ),
]

final_message = llm_with_add_tool.invoke(messages_with_tool_result)
print(final_message.content)

125 + 378 is 503.


## Add a Second Tool

Now we define another mock tool:

```text
multiply_numbers(a, b) -> a * b
```

The model can now choose between addition and multiplication.

In [36]:
@tool
def multiply_numbers(a: float, b: float) -> float:
    """Multiply two numbers and return the result."""
    return a * b


tools = [add_numbers, multiply_numbers]
tool_registry = {tool.name: tool for tool in tools}

llm_with_math_tools = llm.bind_tools(tools)

print("Registered tools:", list(tool_registry))

Registered tools: ['add_numbers', 'multiply_numbers']


## Two-Tool Example

This question needs two steps:

1. Add `12 + 8`.
2. Multiply the result by `3`.

The model may request more than one tool call over multiple model turns.

In [37]:
messages = [
    SystemMessage(content="Use the available math tools. Do not do arithmetic mentally."),
    HumanMessage(content="Calculate (12 + 8) * 3. First add, then multiply."),
]

ai_message = llm_with_math_tools.invoke(messages)

print("First model response content:", ai_message.content)
print("Tool calls:")
print(ai_message.tool_calls)

if not ai_message.tool_calls:
    print("The model answered directly this time. The generic loop below handles repeated tool use more reliably.")

First model response content: 
Tool calls:
[{'name': 'add_numbers', 'args': {'a': 12, 'b': 8}, 'id': 'call_V06safhUsrTVLgRUE9OyjylQ'}, {'name': 'multiply_numbers', 'args': {'a': 20, 'b': 3}, 'id': 'call_nuUcbYJaTLWtVEH1wxU4nA9h'}]


## Helper: Execute Tool Calls

This helper executes every tool call in an `AIMessage`.

It returns LangChain `ToolMessage` objects that can be sent back to the model.

In [38]:
def execute_tool_calls(ai_message):
    """Execute all tool calls in an AIMessage and return ToolMessage objects."""
    tool_messages = []

    for tool_call in ai_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        if tool_name not in tool_registry:
            raise ValueError(f"Unknown tool: {tool_name}")

        selected_tool = tool_registry[tool_name]
        result = selected_tool.invoke(tool_args)

        print("Tool:", tool_name)
        print("Arguments:", tool_args)
        print("Result:", result)
        print()

        tool_messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"],
            )
        )

    return tool_messages

## Generic LangChain Tool-Calling Loop

This is a small agent-style loop.

```text
messages
   |
   v
model with tools
   |
   +-- no tool calls -> final answer
   |
   +-- tool calls
          |
          v
      execute tools
          |
          v
      append ToolMessage results
          |
          v
      call model again
```

LangChain has higher-level agent helpers, but this loop shows the core idea clearly.

In [39]:
def run_langchain_tool_loop(user_question: str, max_turns: int = 5) -> str:
    """Run a simple LangChain tool-calling loop."""
    messages = [
        SystemMessage(
            content=(
                "You are a careful assistant. "
                "Use the available math tools for calculations. "
                "Do not do arithmetic mentally if a tool can do it."
            )
        ),
        HumanMessage(content=user_question),
    ]

    for turn in range(max_turns):
        ai_message = llm_with_math_tools.invoke(messages)
        messages.append(ai_message)

        if not ai_message.tool_calls:
            return ai_message.content

        print(f"Turn {turn + 1}: model requested {len(ai_message.tool_calls)} tool call(s)")

        tool_messages = execute_tool_calls(ai_message)
        messages.extend(tool_messages)

    return "The loop stopped before the model produced a final answer."

In [40]:
answer = run_langchain_tool_loop("Calculate (12 + 8) * 3. First add, then multiply.")

print("Final answer:")
print(answer)

Turn 1: model requested 2 tool call(s)
Tool: add_numbers
Arguments: {'a': 12, 'b': 8}
Result: 20.0

Tool: multiply_numbers
Arguments: {'a': 20, 'b': 3}
Result: 60.0

Final answer:
First, adding 12 and 8 gives 20. Then, multiplying 20 by 3 gives 60. So, (12 + 8) * 3 = 60.


## LangChain Expression Language: LCEL

LangChain has a pipe syntax called LCEL: LangChain Expression Language.

It lets us connect steps with `|`.

```text
prompt | model | parser
```

That means:

```text
format prompt -> call model -> parse output
```

In [44]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a concise instructor for GenAI trainees."),
        ("human", "Explain this LangChain concept in two sentences: {concept}"),
    ]
)

chain = prompt | llm | StrOutputParser()

print(chain.invoke({"concept": "tools"}))

In LangChain, tools are external utilities or APIs that an agent can use to perform specific tasks, like searching the web or accessing a database. They enable the agent to extend its capabilities beyond language understanding by interacting with real-world data or services.


## Build a Small LangChain Chain

Now we build a small training-cost example.

The chain has three parts:

```text
Input values
    |
    v
Create calculation question
    |
    v
Use LangChain tool loop
    |
    v
Summarize result with LCEL
```

In [45]:
summary_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You write clear, short business summaries for training operations."),
        (
            "human",
            "Rewrite this calculation result as a one-sentence business summary:\n\n{calculation_result}",
        ),
    ]
)

summary_chain = summary_prompt | llm | StrOutputParser()


def build_training_cost_question(number_of_trainees: int, hours_per_trainee: int, cost_per_hour: float) -> str:
    return (
        "Calculate the total training cost. "
        f"There are {number_of_trainees} trainees. "
        f"Each trainee attends {hours_per_trainee} hours. "
        f"The cost is {cost_per_hour} per trainee-hour. "
        "First multiply trainees by hours per trainee. "
        "Then multiply that result by cost per hour."
    )


def training_cost_chain(number_of_trainees: int, hours_per_trainee: int, cost_per_hour: float) -> str:
    question = build_training_cost_question(number_of_trainees, hours_per_trainee, cost_per_hour)
    calculation_result = run_langchain_tool_loop(question)
    summary = summary_chain.invoke({"calculation_result": calculation_result})
    return summary


summary = training_cost_chain(
    number_of_trainees=25,
    hours_per_trainee=6,
    cost_per_hour=40,
)

print(summary)

Turn 1: model requested 1 tool call(s)
Tool: multiply_numbers
Arguments: {'a': 25, 'b': 6}
Result: 150.0

Turn 2: model requested 1 tool call(s)
Tool: multiply_numbers
Arguments: {'a': 150, 'b': 40}
Result: 6000.0

The total cost for the training program amounts to $6,000.


## What LangChain Added

Without LangChain, we manually build JSON schemas, parse tool calls, and manage message structures.

With LangChain, we get:

- `ChatOpenAI` for model calls
- `SystemMessage`, `HumanMessage`, and `ToolMessage`
- `@tool` for tool definitions
- `bind_tools(...)` for tool-aware models
- LCEL chains with `prompt | model | parser`
- reusable components that can grow into larger apps

Core idea:

```text
LangChain helps structure GenAI applications.
```

## Key Takeaways

- LangChain is a framework for building LLM applications.
- A LangChain chat model wraps model providers like OpenAI.
- Tools are Python functions the model can request.
- The LLM does not execute tools directly.
- Python executes the tool and returns a `ToolMessage`.
- LCEL lets us compose chains with `|`.
- A tool-calling loop is the foundation of many agent workflows.

## Exercises

### Exercise 1: Add a Subtraction Tool

Create a LangChain tool:

```python
@tool
def subtract_numbers(a: float, b: float) -> float:
    """Subtract b from a and return the result."""
    return a - b
```

Add it to `tools`, rebuild `tool_registry`, and rebind the model.

Test:

```text
What is 500 - 187?
```

### Exercise 2: Add a Division Tool

Create:

```python
@tool
def divide_numbers(a: float, b: float) -> float:
    """Divide a by b and return the result."""
```

Add protection for division by zero.

Test:

```text
What is 144 / 12?
```

### Exercise 3: Add a Mock Business Tool

Create:

```python
@tool
def lookup_training_discount(customer_tier: str) -> float:
    """Return a mock discount rate for a customer tier."""
```

Example behavior:

```text
gold -> 0.20
silver -> 0.10
standard -> 0.00
```

Then update the training-cost chain to calculate discounted cost.

### Exercise 4: Improve Error Handling

Improve `execute_tool_calls(...)` so it handles:

- unknown tools
- invalid tool arguments
- exceptions raised by tools

### Exercise 5: Explain in Your Own Words

Answer:

1. What does `bind_tools(...)` do?
2. Why does Python execute the tool instead of the LLM?
3. What is the difference between a chain and an agent-style loop?
4. Where does the official OpenAI SDK fit when using `ChatOpenAI`?

## Optional Next Steps

After this class, good follow-up topics are:

- LangChain retrievers for RAG
- LangChain agents
- LangGraph for multi-step agent workflows
- structured output
- tracing and observability with LangSmith